In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from scipy import stats

In [2]:
def find_repo_root(start: Path) -> Path:
    """Walk upward from the current directory until the repo root is found."""
    for candidate in [start, *start.parents]:
        if (candidate / "Results_Daily").exists() and (candidate / ".git").exists():
            return candidate
    raise FileNotFoundError("Could not locate repository root from current notebook directory.")


SIGNIFICANCE_LEVEL = 0.10

ASSET_CONFIG = {
    "QQQ": {
        "display_name": "NASDAQ 100",
        "runs": {
            "LSTM-1": "QQQ_LSTM_DAILY_1",
            "LSTM-2": "QQQ_LSTM_DAILY_2",
            "Transformer": "QQQ_TRANSFORMERS_DAILY_1",
            "LSTM-NC-1": "QQQ_LSTM_NC_DAILY",
            "LSTM-NC-2": "QQQ_LSTM_NC_DAILY_2",
        },
    },
    "NKY": {
        "display_name": "NIKKEI 225",
        "runs": {
            "LSTM-1": "NKY_LSTM_DAILY_1",
            "LSTM-2": "NKY_LSTM_DAILY_2",
            "Transformer": "NKY_TRANSFORMERS_DAILY_1",
            "LSTM-NC-1": "NKY_LSTM_NC_DAILY",
            "LSTM-NC-2": "NKY_LSTM_NC_DAILY_2",
        },
    },
    "EUSTX": {
        "display_name": "EURO STOXX 50",
        "runs": {
            "LSTM-1": "EUSTX_LSTM_DAILY_1",
            "LSTM-2": "EUSTX_LSTM_DAILY_2",
            "Transformer": "EUSTX_TRANSFORMERS_DAILY_1",
            "LSTM-NC-1": "EUSTX_LSTM_NC_DAILY",
            "LSTM-NC-2": "EUSTX_LSTM_NC_DAILY_2",
        },
    },
}

# Buy-and-Hold is intentionally excluded per request.
BENCHMARK_COLUMNS = [
    "QQQ Buy-and-Hold"
    # "Equal-Weight Monthly",
    # "Inverse Volatility",
    # "Momentum Top-20%",
    # "Supervised + MVO",
]

NOTEBOOK_DIR = Path.cwd().resolve()
REPO_ROOT = find_repo_root(NOTEBOOK_DIR)
RESULTS_ROOT = REPO_ROOT / "Results_Daily"
OUTPUT_DIR = RESULTS_ROOT / "Statistical_Significance"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repository root: {REPO_ROOT}")
print(f"Results root: {RESULTS_ROOT}")
print(f"Output directory: {OUTPUT_DIR}")

Repository root: /Users/kamilkashif/Documents/University/Masters Thesis/Master_Thesis_DRL
Results root: /Users/kamilkashif/Documents/University/Masters Thesis/Master_Thesis_DRL/Results_Daily
Output directory: /Users/kamilkashif/Documents/University/Masters Thesis/Master_Thesis_DRL/Results_Daily/Statistical_Significance


In [3]:
def load_experiment_frame(asset_code: str, experiment_folder: str) -> pd.DataFrame:
    rl_path = RESULTS_ROOT / asset_code / experiment_folder / "rl_daily_returns_oos.csv"
    benchmark_path = RESULTS_ROOT / asset_code / experiment_folder / "daily_returns_oos.csv"

    rl_df = pd.read_csv(rl_path)
    benchmark_df = pd.read_csv(benchmark_path)

    # Normalize RL return columns across minor file-format differences.
    if "date" not in rl_df.columns:
        rl_df = rl_df.rename(columns={rl_df.columns[0]: "date"})
    if "RL Agent" not in rl_df.columns:
        candidate_cols = [c for c in rl_df.columns if c != "date"]
        if not candidate_cols:
            raise ValueError(f"No return column found in {rl_path}")
        rl_df = rl_df.rename(columns={candidate_cols[0]: "RL Agent"})

    rl_df["date"] = pd.to_datetime(rl_df["date"])
    benchmark_df["date"] = pd.to_datetime(benchmark_df["date"])

    use_cols = ["date", *BENCHMARK_COLUMNS]
    merged = (
        rl_df[["date", "RL Agent"]]
        .rename(columns={"RL Agent": "strategy_return"})
        .merge(benchmark_df[use_cols], on="date", how="inner")
        .dropna(subset=["strategy_return", *BENCHMARK_COLUMNS])
        .sort_values("date")
        .reset_index(drop=True)
    )

    return merged


experiment_data_by_asset = {}
coverage_rows = []

for asset_code, asset_cfg in ASSET_CONFIG.items():
    experiment_data_by_asset[asset_code] = {}
    for model_label, folder_name in asset_cfg["runs"].items():
        df_model = load_experiment_frame(asset_code, folder_name)
        experiment_data_by_asset[asset_code][model_label] = df_model

        coverage_rows.append(
            {
                "Asset": asset_cfg["display_name"],
                "Model": model_label,
                "N": len(df_model),
                "Start": df_model["date"].min().date(),
                "End": df_model["date"].max().date(),
            }
        )

coverage_df = pd.DataFrame(coverage_rows)
display(coverage_df.sort_values(["Asset", "Model"]).reset_index(drop=True))

,Asset,Model,N,Start,End
0,EURO STOXX 50,LSTM-1,4085,2009-04-01,2025-03-28
1,EURO STOXX 50,LSTM-2,4085,2009-04-01,2025-03-28
2,EURO STOXX 50,LSTM-NC-1,4085,2009-04-01,2025-03-28
3,EURO STOXX 50,LSTM-NC-2,4085,2009-04-01,2025-03-28
4,EURO STOXX 50,Transformer,4085,2009-04-01,2025-03-28
5,NASDAQ 100,LSTM-1,4008,2009-04-06,2025-04-01
6,NASDAQ 100,LSTM-2,4008,2009-04-06,2025-04-01
7,NASDAQ 100,LSTM-NC-1,4008,2009-04-06,2025-04-01
8,NASDAQ 100,LSTM-NC-2,4008,2009-04-06,2025-04-01
9,NASDAQ 100,Transformer,4008,2009-04-06,2025-04-01


In [4]:
paired_rows = []

for asset_code, asset_cfg in ASSET_CONFIG.items():
    for model_label, df_model in experiment_data_by_asset[asset_code].items():
        for benchmark in BENCHMARK_COLUMNS:
            aligned = df_model[["strategy_return", benchmark]].dropna()
            test_result = stats.ttest_rel(
                aligned["strategy_return"],
                aligned[benchmark],
                alternative="greater",
            )

            is_significant = bool(test_result.pvalue < SIGNIFICANCE_LEVEL)
            paired_rows.append(
                {
                    "Asset": asset_cfg["display_name"],
                    "Asset Code": asset_code,
                    "Model": model_label,
                    "Benchmark": benchmark,
                    "N": len(aligned),
                    "Mean(strategy - benchmark)": (
                        aligned["strategy_return"] - aligned[benchmark]
                    ).mean(),
                    "t_stat": test_result.statistic,
                    "p_value": test_result.pvalue,
                    "Significant": "Yes" if is_significant else "No",
                    "Significant_10pct": is_significant,
                }
            )

paired_results = pd.DataFrame(paired_rows).sort_values(
    ["Asset", "Model", "Benchmark"]
).reset_index(drop=True)

paired_results.head()

,Asset,Asset Code,Model,Benchmark,N,Mean(strategy - benchmark),t_stat,p_value,Significant,Significant_10pct
0,EURO STOXX 50,EUSTX,LSTM-1,QQQ Buy-and-Hold,4085,2.236128e-07,0.001407,0.499439,No,False
1,EURO STOXX 50,EUSTX,LSTM-2,QQQ Buy-and-Hold,4085,-2.471463e-05,-0.155081,0.561617,No,False
2,EURO STOXX 50,EUSTX,LSTM-NC-1,QQQ Buy-and-Hold,4085,-3.938654e-05,-0.232234,0.591816,No,False
3,EURO STOXX 50,EUSTX,LSTM-NC-2,QQQ Buy-and-Hold,4085,9.469452e-06,0.058269,0.476768,No,False
4,EURO STOXX 50,EUSTX,Transformer,QQQ Buy-and-Hold,4085,6.649134e-06,0.041772,0.483341,No,False


In [5]:
paired_results

,Asset,Asset Code,Model,Benchmark,N,Mean(strategy - benchmark),t_stat,p_value,Significant,Significant_10pct
0,EURO STOXX 50,EUSTX,LSTM-1,QQQ Buy-and-Hold,4085,2.236128e-07,0.001407,0.499439,No,False
1,EURO STOXX 50,EUSTX,LSTM-2,QQQ Buy-and-Hold,4085,-2.471463e-05,-0.155081,0.561617,No,False
2,EURO STOXX 50,EUSTX,LSTM-NC-1,QQQ Buy-and-Hold,4085,-3.938654e-05,-0.232234,0.591816,No,False
3,EURO STOXX 50,EUSTX,LSTM-NC-2,QQQ Buy-and-Hold,4085,9.469452e-06,0.058269,0.476768,No,False
4,EURO STOXX 50,EUSTX,Transformer,QQQ Buy-and-Hold,4085,6.649134e-06,0.041772,0.483341,No,False
5,NASDAQ 100,QQQ,LSTM-1,QQQ Buy-and-Hold,4008,-3.726104e-05,-0.404139,0.656934,No,False
6,NASDAQ 100,QQQ,LSTM-2,QQQ Buy-and-Hold,4008,-1.282242e-04,-1.554796,0.939963,No,False
7,NASDAQ 100,QQQ,LSTM-NC-1,QQQ Buy-and-Hold,4008,5.612736e-05,0.467978,0.319913,No,False
8,NASDAQ 100,QQQ,LSTM-NC-2,QQQ Buy-and-Hold,4008,1.999502e-06,0.020336,0.491888,No,False
9,NASDAQ 100,QQQ,Transformer,QQQ Buy-and-Hold,4008,-4.423643e-05,-0.484329,0.685911,No,False


In [10]:
def bold_if_significant(value: float) -> str:
    if pd.notna(value) and value < SIGNIFICANCE_LEVEL:
        return "font-weight: bold"
    return ""


print("Paired t-test p-values by asset (right-tailed, H1: strategy > benchmark)")

for asset_code, asset_cfg in ASSET_CONFIG.items():
    asset_name = asset_cfg["display_name"]
    print(f"\n{asset_name}")

    asset_slice = paired_results.loc[paired_results["Asset Code"] == asset_code]

    pvalue_matrix = asset_slice.pivot(
        index="Model",
        columns="Benchmark",
        values="p_value",
    ).reindex(index=list(asset_cfg["runs"].keys()), columns=BENCHMARK_COLUMNS)

    # display(pvalue_matrix.style.format("{:.4f}").map(bold_if_significant))

    display(
        asset_slice[
            [
                "Model",
                "Benchmark",
                "N",
                "Mean(strategy - benchmark)",
                "t_stat",
                "p_value",
                "Significant",
            ]
        ]
        .sort_values(["Model", "Benchmark"])
        .reset_index(drop=True)
    )

Paired t-test p-values by asset (right-tailed, H1: strategy > benchmark)

NASDAQ 100


,Model,Benchmark,N,Mean(strategy - benchmark),t_stat,p_value,Significant
0,LSTM-1,QQQ Buy-and-Hold,4008,-0.000037,-0.404139,0.656934,No
1,LSTM-2,QQQ Buy-and-Hold,4008,-0.000128,-1.554796,0.939963,No
2,LSTM-NC-1,QQQ Buy-and-Hold,4008,0.000056,0.467978,0.319913,No
3,LSTM-NC-2,QQQ Buy-and-Hold,4008,0.000002,0.020336,0.491888,No
4,Transformer,QQQ Buy-and-Hold,4008,-0.000044,-0.484329,0.685911,No



NIKKEI 225


,Model,Benchmark,N,Mean(strategy - benchmark),t_stat,p_value,Significant
0,LSTM-1,QQQ Buy-and-Hold,3916,4.399317e-05,0.398340,0.345201,No
1,LSTM-2,QQQ Buy-and-Hold,3916,-5.075688e-05,-0.493750,0.689245,No
2,LSTM-NC-1,QQQ Buy-and-Hold,3916,-1.946863e-07,-0.001508,0.500602,No
3,LSTM-NC-2,QQQ Buy-and-Hold,3916,-7.378108e-07,-0.006046,0.502412,No
4,Transformer,QQQ Buy-and-Hold,3916,-2.810061e-05,-0.249059,0.598336,No



EURO STOXX 50


,Model,Benchmark,N,Mean(strategy - benchmark),t_stat,p_value,Significant
0,LSTM-1,QQQ Buy-and-Hold,4085,2.236128e-07,0.001407,0.499439,No
1,LSTM-2,QQQ Buy-and-Hold,4085,-2.471463e-05,-0.155081,0.561617,No
2,LSTM-NC-1,QQQ Buy-and-Hold,4085,-3.938654e-05,-0.232234,0.591816,No
3,LSTM-NC-2,QQQ Buy-and-Hold,4085,9.469452e-06,0.058269,0.476768,No
4,Transformer,QQQ Buy-and-Hold,4085,6.649134e-06,0.041772,0.483341,No


In [7]:
regression_rows = []

for asset_code, asset_cfg in ASSET_CONFIG.items():
    for model_label, df_model in experiment_data_by_asset[asset_code].items():
        for benchmark in BENCHMARK_COLUMNS:
            aligned = df_model[["strategy_return", benchmark]].dropna().copy()

            y = (aligned["strategy_return"] - aligned[benchmark]).to_numpy(dtype=float)
            x = aligned[benchmark].to_numpy(dtype=float)
            n = len(aligned)

            if n <= 2:
                raise ValueError(
                    f"Not enough observations for regression: {asset_code} {model_label} vs {benchmark}"
                )

            x_mean = x.mean()
            y_mean = y.mean()
            x_centered = x - x_mean
            y_centered = y - y_mean

            sxx = np.sum(x_centered**2)
            sxy = np.sum(x_centered * y_centered)

            if np.isclose(sxx, 0.0):
                raise ValueError(f"Benchmark variance is zero: {asset_code} {model_label} vs {benchmark}")

            beta = sxy / sxx
            alpha = y_mean - beta * x_mean

            y_hat = alpha + beta * x
            resid = y - y_hat

            sse = np.sum(resid**2)
            sst = np.sum((y - y_mean) ** 2)
            df_resid = n - 2
            sigma2 = sse / df_resid

            se_beta = np.sqrt(sigma2 / sxx)
            se_alpha = np.sqrt(sigma2 * (1.0 / n + (x_mean**2) / sxx))

            t_alpha = alpha / se_alpha if se_alpha > 0 else np.nan
            t_beta = beta / se_beta if se_beta > 0 else np.nan

            p_alpha_right = 1.0 - stats.t.cdf(t_alpha, df=df_resid) if np.isfinite(t_alpha) else np.nan
            p_beta_two_sided = (
                2.0 * (1.0 - stats.t.cdf(abs(t_beta), df=df_resid)) if np.isfinite(t_beta) else np.nan
            )

            r2 = 1.0 - (sse / sst) if not np.isclose(sst, 0.0) else np.nan
            is_significant = bool(p_alpha_right < SIGNIFICANCE_LEVEL) if pd.notna(p_alpha_right) else False

            regression_rows.append(
                {
                    "Asset": asset_cfg["display_name"],
                    "Asset Code": asset_code,
                    "Model": model_label,
                    "Benchmark": benchmark,
                    "N": n,
                    "alpha": alpha,
                    "SE(alpha)": se_alpha,
                    "t(alpha)": t_alpha,
                    "p(alpha) right-tail": p_alpha_right,
                    "beta": beta,
                    "SE(beta)": se_beta,
                    "t(beta)": t_beta,
                    "p(beta) two-sided": p_beta_two_sided,
                    "R2": r2,
                    "Significant": "Yes" if is_significant else "No",
                    "Significant_10pct": is_significant,
                }
            )

regression_results = pd.DataFrame(regression_rows).sort_values(
    ["Asset", "Model", "Benchmark"]
).reset_index(drop=True)

regression_results.head()

,Asset,Asset Code,Model,Benchmark,N,alpha,SE(alpha),t(alpha),p(alpha) right-tail,beta,SE(beta),t(beta),p(beta) two-sided,R2,Significant,Significant_10pct
0,EURO STOXX 50,EUSTX,LSTM-1,QQQ Buy-and-Hold,4085,0.000163,0.000124,1.317944,0.093798,-0.421679,0.008221,-51.295149,0.0,0.391885,Yes,True
1,EURO STOXX 50,EUSTX,LSTM-2,QQQ Buy-and-Hold,4085,0.000171,0.000106,1.618146,0.052854,-0.505843,0.007008,-72.179769,0.0,0.560633,Yes,True
2,EURO STOXX 50,EUSTX,LSTM-NC-1,QQQ Buy-and-Hold,4085,0.000121,0.000139,0.871027,0.191895,-0.414051,0.009198,-45.013775,0.0,0.331668,No,False
3,EURO STOXX 50,EUSTX,LSTM-NC-2,QQQ Buy-and-Hold,4085,0.000166,0.000132,1.262752,0.103375,-0.404842,0.008723,-46.408893,0.0,0.345336,No,False
4,EURO STOXX 50,EUSTX,Transformer,QQQ Buy-and-Hold,4085,0.000169,0.000125,1.357387,0.087367,-0.420036,0.008265,-50.823418,0.0,0.387491,Yes,True


In [11]:
regression_results

,Asset,Asset Code,Model,Benchmark,N,alpha,SE(alpha),t(alpha),p(alpha) right-tail,beta,SE(beta),t(beta),p(beta) two-sided,R2,Significant,Significant_10pct
0,EURO STOXX 50,EUSTX,LSTM-1,QQQ Buy-and-Hold,4085,1.633835e-04,0.000124,1.317944,0.093798,-0.421679,0.008221,-51.295149,0.000000e+00,0.391885,Yes,True
1,EURO STOXX 50,EUSTX,LSTM-2,QQQ Buy-and-Hold,4085,1.710105e-04,0.000106,1.618146,0.052854,-0.505843,0.007008,-72.179769,0.000000e+00,0.560633,Yes,True
2,EURO STOXX 50,EUSTX,LSTM-NC-1,QQQ Buy-and-Hold,4085,1.208219e-04,0.000139,0.871027,0.191895,-0.414051,0.009198,-45.013775,0.000000e+00,0.331668,No,False
3,EURO STOXX 50,EUSTX,LSTM-NC-2,QQQ Buy-and-Hold,4085,1.661146e-04,0.000132,1.262752,0.103375,-0.404842,0.008723,-46.408893,0.000000e+00,0.345336,No,False
4,EURO STOXX 50,EUSTX,Transformer,QQQ Buy-and-Hold,4085,1.691733e-04,0.000125,1.357387,0.087367,-0.420036,0.008265,-50.823418,0.000000e+00,0.387491,Yes,True
5,NASDAQ 100,QQQ,LSTM-1,QQQ Buy-and-Hold,4008,-4.919582e-06,0.000092,-0.053481,0.521324,-0.041691,0.007141,-5.838331,5.690588e-09,0.008437,No,False
6,NASDAQ 100,QQQ,LSTM-2,QQQ Buy-and-Hold,4008,-9.841162e-07,0.000076,-0.013020,0.505194,-0.164024,0.005868,-27.953465,0.000000e+00,0.163219,No,False
7,NASDAQ 100,QQQ,LSTM-NC-1,QQQ Buy-and-Hold,4008,3.401826e-05,0.000120,0.283418,0.388436,0.028501,0.009318,3.058741,2.237382e-03,0.002330,No,False
8,NASDAQ 100,QQQ,LSTM-NC-2,QQQ Buy-and-Hold,4008,3.880667e-06,0.000099,0.039393,0.484290,-0.002425,0.007647,-0.317096,7.511870e-01,0.000025,No,False
9,NASDAQ 100,QQQ,Transformer,QQQ Buy-and-Hold,4008,-5.409269e-06,0.000091,-0.059479,0.523713,-0.050052,0.007060,-7.089534,1.583178e-12,0.012391,No,False


In [12]:
print("Linear regression results by asset (alpha right-tailed test, H1: alpha > 0)")

for asset_code, asset_cfg in ASSET_CONFIG.items():
    asset_name = asset_cfg["display_name"]
    print(f"\n{asset_name}")

    asset_slice = regression_results.loc[regression_results["Asset Code"] == asset_code]

    display(
        asset_slice[
            [
                "Model",
                "Benchmark",
                "N",
                "alpha",
                "SE(alpha)",
                "t(alpha)",
                "p(alpha) right-tail",
                "beta",
                "SE(beta)",
                "t(beta)",
                "p(beta) two-sided",
                "R2",
                "Significant",
            ]
        ]
        .sort_values(["Model", "Benchmark"]) 
        .style
        .format(
            {
                "alpha": "{:.6f}",
                "SE(alpha)": "{:.6f}",
                "t(alpha)": "{:.4f}",
                "p(alpha) right-tail": "{:.4f}",
                "beta": "{:.6f}",
                "SE(beta)": "{:.6f}",
                "t(beta)": "{:.4f}",
                "p(beta) two-sided": "{:.4f}",
                "R2": "{:.4f}",
            }
        )
        .map(
            lambda v: "font-weight: bold"
            if pd.notna(v) and isinstance(v, (float, np.floating)) and v < SIGNIFICANCE_LEVEL
            else "",
            subset=["p(alpha) right-tail"],
        )
    )

# Save outputs
paired_results.to_csv(OUTPUT_DIR / "paired_ttest_all_assets.csv", index=False)
regression_results.to_csv(OUTPUT_DIR / "alpha_regression_all_assets.csv", index=False)

for asset_code, asset_cfg in ASSET_CONFIG.items():
    asset_slug = asset_cfg["display_name"].lower().replace(" ", "_")

    paired_results.loc[paired_results["Asset Code"] == asset_code].to_csv(
        OUTPUT_DIR / f"paired_ttest_{asset_slug}.csv", index=False
    )
    regression_results.loc[regression_results["Asset Code"] == asset_code].to_csv(
        OUTPUT_DIR / f"alpha_regression_{asset_slug}.csv", index=False
    )

print(f"\nSaved results to: {OUTPUT_DIR}")

Linear regression results by asset (alpha right-tailed test, H1: alpha > 0)

NASDAQ 100


,Model,Benchmark,N,alpha,SE(alpha),t(alpha),p(alpha) right-tail,beta,SE(beta),t(beta),p(beta) two-sided,R2,Significant
5,LSTM-1,QQQ Buy-and-Hold,4008,-0.000005,0.000092,-0.0535,0.5213,-0.041691,0.007141,-5.8383,0.0000,0.0084,No
6,LSTM-2,QQQ Buy-and-Hold,4008,-0.000001,0.000076,-0.0130,0.5052,-0.164024,0.005868,-27.9535,0.0000,0.1632,No
7,LSTM-NC-1,QQQ Buy-and-Hold,4008,0.000034,0.000120,0.2834,0.3884,0.028501,0.009318,3.0587,0.0022,0.0023,No
8,LSTM-NC-2,QQQ Buy-and-Hold,4008,0.000004,0.000099,0.0394,0.4843,-0.002425,0.007647,-0.3171,0.7512,0.0000,No
9,Transformer,QQQ Buy-and-Hold,4008,-0.000005,0.000091,-0.0595,0.5237,-0.050052,0.007060,-7.0895,0.0000,0.0124,No



NIKKEI 225


,Model,Benchmark,N,alpha,SE(alpha),t(alpha),p(alpha) right-tail,beta,SE(beta),t(beta),p(beta) two-sided,R2,Significant
10,LSTM-1,QQQ Buy-and-Hold,3916,0.000110,0.000106,1.0349,0.1504,-0.141810,0.007996,-17.7345,0.0000,0.0744,No
11,LSTM-2,QQQ Buy-and-Hold,3916,0.000060,0.000090,0.6718,0.2509,-0.238156,0.006735,-35.3621,0.0000,0.2421,No
12,LSTM-NC-1,QQQ Buy-and-Hold,3916,0.000014,0.000129,0.1051,0.4581,-0.029534,0.009702,-3.0441,0.0023,0.0024,No
13,LSTM-NC-2,QQQ Buy-and-Hold,3916,0.000015,0.000122,0.1195,0.4524,-0.032866,0.009168,-3.5847,0.0003,0.0033,No
14,Transformer,QQQ Buy-and-Hold,3916,0.000029,0.000110,0.2659,0.3951,-0.123049,0.008260,-14.8971,0.0000,0.0537,No



EURO STOXX 50


,Model,Benchmark,N,alpha,SE(alpha),t(alpha),p(alpha) right-tail,beta,SE(beta),t(beta),p(beta) two-sided,R2,Significant
0,LSTM-1,QQQ Buy-and-Hold,4085,0.000163,0.000124,1.3179,0.0938,-0.421679,0.008221,-51.2951,0.0000,0.3919,Yes
1,LSTM-2,QQQ Buy-and-Hold,4085,0.000171,0.000106,1.6181,0.0529,-0.505843,0.007008,-72.1798,0.0000,0.5606,Yes
2,LSTM-NC-1,QQQ Buy-and-Hold,4085,0.000121,0.000139,0.8710,0.1919,-0.414051,0.009198,-45.0138,0.0000,0.3317,No
3,LSTM-NC-2,QQQ Buy-and-Hold,4085,0.000166,0.000132,1.2628,0.1034,-0.404842,0.008723,-46.4089,0.0000,0.3453,No
4,Transformer,QQQ Buy-and-Hold,4085,0.000169,0.000125,1.3574,0.0874,-0.420036,0.008265,-50.8234,0.0000,0.3875,Yes



Saved results to: /Users/kamilkashif/Documents/University/Masters Thesis/Master_Thesis_DRL/Results_Daily/Statistical_Significance
